In [ ]:
# Configure ADLS access through the workspace before running this notebook.
storage_account_name = "YOUR_STORAGE_ACCOUNT"


In [ ]:
enriched_reviews_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/enriched_reviews/"
enriched_df = spark.read.parquet(enriched_reviews_path)


In [ ]:
from pyspark.sql.functions import col, year, to_date, regexp_replace, trim

# Amazon's reviewTime uses dates such as "07 23, 2014".
enriched_df = enriched_df.withColumn(
    "review_year",
    year(to_date(regexp_replace(trim(col("reviewTime")), r"\s+", " "), "MM d, yyyy"))
)
features_v1_df = enriched_df.select(
    "asin", "reviewerID", "overall", "summary", "reviewText", "helpful",
    "reviewTime", "review_year", "title", "brand", "price"
)


In [ ]:
gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/features_v1/"
features_v1_df.write.mode("overwrite").parquet(gold_path)


In [ ]:
gold_df = spark.read.parquet(gold_path)
gold_df.printSchema()
display(gold_df.limit(5))
print("Final row count:", gold_df.count())